In [1]:
from pathlib import Path
import spacy

class NER:
	def __init__(self, model_names=["en_ner_jnlpba_md", "en_ner_bc5cdr_md", "en_ner_bionlp13cg_md", "../graph_src/health_ner"]):
		print("Loading NER models")
		self.models = [spacy.load(name) for name in model_names]
		print("NER models loaded successfully")

		self.label_map = {
			"ANATOMICAL_SYSTEM": "anatomy",
			"CELL": "anatomy",
			"CELLULAR_COMPONENT": "cellular_component",
			"DEVELOPING_ANATOMICAL_STRUCTURE": "anatomy",
			"IMMATERIAL_ANATOMICAL_ENTITY": "anatomy",
			"MULTI-TISSUE_STRUCTURE": "anatomy",
			"ORGAN": "anatomy",
			"ORGANISM_SUBDIVISION": "anatomy",
			"ORGANISM_SUBSTANCE": "anatomy",
			"PATHOLOGICAL_FORMATION": "anatomy",
			"TISSUE": "anatomy",

			"GENE_OR_GENE_PRODUCT": "gene_protein",
			"PROTEIN": "gene_protein",
			"DNA": "gene_protein",
			"RNA": "gene_protein",

			"DISEASE": "disease",
			"CANCER": "disease",

			"CHEMICAL": "drug",
			"SIMPLE_CHEMICAL": "drug",

			# Not caught by ordinary NER (caught by my ruler!)
			"DRUG": "drug",
			"BIOLOGICAL_PROCESS": "biological_process",
			"MOLECULAR_FUNCTION": "molecular_function",
			"PATHWAY": "pathway",
			"EFFECT_PHENOTYPE": "effect_phenotype",
			"EXPOSURE": "exposure",
		}

		self.target_labels = set(self.label_map.values())

	def find(self, query, subset_deleter=True):
		all_entities = []
		for ner in self.models:
			doc = ner(query)
			for ent in doc.ents:
				mapped_label = self.label_map.get(ent.label_)
				if mapped_label in self.target_labels:
					all_entities.append((ent.text.lower().strip(), mapped_label, ent.start_char, ent.end_char))

		seen = set()
		unique_entities = []

		for text, label, start, end in all_entities:
			if text not in seen:
				seen.add(text)
				unique_entities.append({
					"text": text,
					"label": label,
					"start": start,
					"end": end
				})

		if subset_deleter:
			unique_entities.sort(key=lambda e: e["end"] - e["start"], reverse=True)
			filtered = []
			for candidate in unique_entities:
				is_subspan = False
				for kept in filtered:
					if (candidate["start"] >= kept["start"] and candidate["end"] <= kept["end"]):
						is_subspan = True
						break
				if not is_subspan:
					filtered.append(candidate)
			unique_entities = filtered
		
		return unique_entities

In [2]:
ner = NER()

Loading NER models


/home/marco/miniconda3/envs/GraphMedRag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/marco/miniconda3/envs/GraphMedRag/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


NER models loaded successfully


In [9]:
import json

with open("../benchmark_jsons/benchmark_100.json", "r") as f:
	data = json.load(f)

questions = []

for item in data["bioasq"].values():
	question = item["question"]
	options = item["options"]
	
	full_text = question + "\n"
	for key, value in options.items():
		full_text += f"{value}\n"
	
	questions.append(full_text)

In [10]:
results = []

for q in questions:
	results.append(ner.find(q, True))

In [11]:
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from rapidfuzz import process, fuzz
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import joblib
import faiss
import torch
import json

class EntityLinker:
	def __init__(
		self,
		device,
		HNSW=False,
		M=32,
		batch_size=2048,
		faiss_index_dir= "data/entity-faiss.index",
		nodes_csv= "data/nodes.csv",
		embeddings_file= "data/nodes_embeddings_98-315.csv",
		pca_file= "data/pca_model-315.joblib",
		llm_name="ncbi/MedCPT-Query-Encoder",
	):
		
		self.device = "cpu" if device=="cpu" else "cuda"
		self.tokenizer = AutoTokenizer.from_pretrained(llm_name)
		self.model = AutoModel.from_pretrained(llm_name)
		self.model.to(self.device)
		self.model.eval()

		self.save_data_folder = "data"
		self.df = pd.read_csv(nodes_csv, usecols=['node_index', 'node_type', 'node_name'])
		self.idxs = self.df['node_index'].tolist()
		self.types = self.df['node_type'].tolist()
		self.names = self.df['node_name'].tolist()

		self.perc_components = 0.98

		if not Path(embeddings_file).exists():
			# Crea embeddings
			all_embeddings = self.embedder(batch_size)
			# Carica/Calcola la PCA
			self.pca = self.pca_compute_or_load(pca_file, all_embeddings)

			embeddings_reduced = self.pca.transform(all_embeddings)
			embeddings_str = [json.dumps(vec.tolist()) for vec in embeddings_reduced]
			dimensions = [str(self.perc_components).split(".")[1], str(embeddings_reduced.shape[1])]
			# print("Shape embeddings ridotti:", dimensions[1])

			output_df = pd.DataFrame({
				"node_index": self.idxs,
				"node_type": self.types,
				"node_name": self.names,
				"embedding": embeddings_str
			})
			# Salva gli Embeddings
			embeddings_file = self.save_node_embeddings(output_df, dimensions)
		else:
			# Carica gli embeddings
			all_embeddings = self.embeddings_loader(embeddings_file)
			
			# Carica/Calcola la PCA
			self.pca = self.pca_compute_or_load(pca_file, all_embeddings)
			# Estrai numero tra l'ultimo "_" e "-", e tra "-" e ".csv"
			filename = Path(embeddings_file).name
			name_no_ext = Path(filename).stem
			
			last_part = name_no_ext.split("_")[-1]
			dim0, dim1 = last_part.split("-")
			dimensions = [dim0, dim1]
			
		if not Path(pca_file).exists():
			# Salva la PCA
			self.save_pca(self.pca, dimensions)

		if not Path(faiss_index_dir).exists():
			self.index = self.construct_index(embeddings_file, h_dim=int(dimensions[1]), HNSW=HNSW, M=M)
		else:
			print("Loading FAISS Index")
			self.index = faiss.read_index(str(faiss_index_dir))

	# Se prendi più di un FUZZY dopo il codice esplode ! ! !
	def link(self, query_word, fuzzy_f=1, sim_k=30, white_spaces=False):
		query_word = query_word["text"]
		embedding = self.create_embeddings(query_word)
		embeddings_reduced = self.pca.transform(embedding)

		faiss.normalize_L2(embeddings_reduced)
		hits = self.index.search(embeddings_reduced, k=sim_k)
		top_k = self.df.iloc[hits[1][0]]

		database_strings = top_k['node_name'].tolist()

		# print(database_strings)
		if white_spaces:
			query_word = query_word.replace(' ', '')
			database_strings = [i.replace(' ', '') for i in database_strings] # nel caso volessi usare metodo senza spazi

		# If query_word == una sola parola uso Levenshtein distance
		scorer = fuzz.ratio if ' ' not in query_word and not white_spaces else fuzz.WRatio

		fuzzy_candidate = process.extract(
			query_word,
			database_strings,
			scorer=scorer,
			limit=fuzzy_f
		)

		### Codice che poi riordina in base all'ordine della Cosine Similarity (non proprio quello che voglio -.-) !!!!!
		fuzzy_candidate = [x[0] for x in fuzzy_candidate]
		# print(fuzzy_candidate)
		results_df = top_k[top_k['node_name'].isin(fuzzy_candidate)]

		subset_df = results_df[['node_index', 'node_type', 'node_name']]
		result = subset_df.values.squeeze().tolist()

		print("FUZZY: ", fuzzy_candidate)
		print(query_word, ":")
		print(", ".join(database_strings), "\n")

		return result
	
	def embedder(self, batch_size):
		print("Creating Embeddings")
		all_embeddings = []
		for i in tqdm(range(0, len(self.names), batch_size), desc="Calcolo embedding", ncols=100):
			batch_names = self.names[i:i+batch_size]
			batch_embeddings = self.create_embeddings(batch_names)
			
			all_embeddings.append(batch_embeddings)

		all_embeddings = np.vstack(all_embeddings)
		# print("Shape embeddings originali:", all_embeddings.shape)

		return all_embeddings

	def create_embeddings(self, words):
		inputs = self.tokenizer(words, padding=True, truncation=False, return_tensors="pt").to(self.device)
		
		with torch.no_grad():
			outputs = self.model(**inputs)
			last_hidden_state = outputs.last_hidden_state
			# print(words, last_hidden_state.shape)
			batch_embeddings = last_hidden_state.mean(dim=1).cpu().numpy()
			# takes all the embedding values obtained by tokenization of a word/sentence and compute the mean of them

		return batch_embeddings
	
	def construct_index(self, embedding_csv, h_dim, HNSW=False, M=32):
		print("Building FAISS Index")
		if HNSW:
			index = faiss.IndexHNSWFlat(h_dim, M)
			index.metric_type = faiss.METRIC_INNER_PRODUCT
		else:
			index = faiss.IndexFlatIP(h_dim)

		df = pd.read_csv(embedding_csv)

		embeddings = df['embedding'].apply(lambda x: np.array(eval(x), dtype='float32'))
		embeddings_matrix = np.stack(embeddings.to_numpy())

		faiss.normalize_L2(embeddings_matrix)
		index.add(embeddings_matrix)
		faiss.write_index(index, str(self.save_data_folder / "entity-faiss.index"))

		print(f"Faiss index built with {index.ntotal} vectors")
		
		return index
	
	def save_pca(self, pca, dimensions):
		print("Saving PCA on file")
		output_path = self.save_data_folder / ("pca_model-" + dimensions[1] + ".joblib")
		joblib.dump(pca, output_path)
		print("PCA file saved in: " + str(output_path))

	def pca_compute_or_load(self, pca_file, all_embeddings):
		if not Path(pca_file).exists():
			print("Computing PCA")
			pca = PCA(n_components=self.perc_components)
			pca.fit(all_embeddings)
		else:
			print("Loading PCA")
			pca = joblib.load(pca_file)
		
		return pca

	def save_node_embeddings(self, output_df, dimensions):
		print("Saving Embeddings on file")
		output_csv = "nodes_embeddings_" + dimensions[0] + "-" + dimensions[1] + ".csv"
		output_path = self.save_data_folder / output_csv
		output_df.to_csv(output_path, index=False)
		print(f"Node Embeddings CSV saved in: {output_path}")
		
		return output_path
	
	def embeddings_loader(self, embeddings_file):
		print("Loading Embeddings")
		df = pd.read_csv(embeddings_file)
		embeddings = df['embedding'].apply(lambda x: np.array(json.loads(x), dtype='float32'))
		embeddings = np.stack(embeddings.to_numpy())

		return embeddings

In [12]:
entity_linker = EntityLinker("cuda")

Loading Embeddings
Loading PCA
Loading FAISS Index


In [13]:
from itertools import combinations

all_link = []

for i, result in enumerate(results):
	linked_entities = set()

	for entity in result:
		link_result = entity_linker.link(entity)

		if isinstance(link_result, list) and all(isinstance(r, list) for r in link_result):
			for r in link_result:
				linked_entities.add(tuple(r))
		else:
			linked_entities.add(tuple(link_result))
"""
	linked_entities = list(linked_entities)
	pairs = list(combinations(linked_entities, 2))
	pairs = [(a, b) for (a, b) in pairs if a[2] != b[2]]

	if len(pairs) > 0:
		print("Domanda", str(i + 1))
		print(questions[i])
		for entity in result:
			link_result = entity_linker.link2(entity)
			print("RRRRRRRRRRRRRRRRRRRRRRRR: ", link_result, "\n")
		#[print(s, t) for s, t in pairs]
		print()
	all_link.append(pairs)"""

FUZZY:  ['Alzheimer disease']
alzheimer's disease :
Alzheimer disease, Alzheimer disease without neurofibrillary tangles, familial Alzheimer disease, dementia (disease), Dementia, neurodegenerative disease, Neurodegenerative Diseases, metabolic disease with dementia, tauopathy, Neurofibrillary tangles, dementia/parkinsonism with non-Alzheimer amyloid plaques, genetic dementia, neurofibrillary tangle, cerebrovascular dementia, brain disease, vascular dementia, senile degeneration of brain, AIDS dementia complex, ABeta amyloidosis, Senile plaques, Alzheimer disease, susceptibility to, mitochondrial, Generalized amyloid deposition, inherited neurodegenerative disorder, Neurodegeneration, Cerebral amyloid angiopathy, cerebral lipidosis with dementia, amyloid-beta formation, cerebral amyloid angiopathy, parkinsonian syndrome due to neurodegenerative disease, Alzheimer disease, familial early-onset, with coexisting amyloid and prion pathology 

FUZZY:  ['Brain atrophy']
brain atrophy :
Brain

'\n\tlinked_entities = list(linked_entities)\n\tpairs = list(combinations(linked_entities, 2))\n\tpairs = [(a, b) for (a, b) in pairs if a[2] != b[2]]\n\n\tif len(pairs) > 0:\n\t\tprint("Domanda", str(i + 1))\n\t\tprint(questions[i])\n\t\tfor entity in result:\n\t\t\tlink_result = entity_linker.link2(entity)\n\t\t\tprint("RRRRRRRRRRRRRRRRRRRRRRRR: ", link_result, "\n")\n\t\t#[print(s, t) for s, t in pairs]\n\t\tprint()\n\tall_link.append(pairs)'